In [1]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import matplotlib.pyplot as plt
import pandas as pd
import math

# ---------------------------- Run from Repo Root ----------------------------
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

# ---------------------------- File Paths ----------------------------
country_list_csv = BASE_DIR / "data" / "STANDARD_COUNTRY_LIST.csv"
CR_Box_Countries = BASE_DIR / "data" / "CR_Box_Countries_MS.csv"
ecw_country = BASE_DIR / "results" / "ECWbyCountry.csv"
Baghouse_Airflow = BASE_DIR /'data'/"BaghouseAirflow.csv"

from country_pkg import Country

In [2]:
# ------------------------ Functions ----------------------------

def generate_countries_from_multiple_csvs(
    country_csv_path,
    cr_box_csv_path=None,
    ecw_csv_path=None,
    baghouse_csv_path=None
):
    # ---------------- Main country CSV ----------------
    df = pd.read_csv(country_csv_path, encoding='cp1252')
    required_cols = ['ISO-3', 'Country Name']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must have a column named '{col}'")
    
    # ---------------- CR Box CSV ----------------
    cr_box_df = None
    if cr_box_csv_path:
        cr_box_df = pd.read_csv(cr_box_csv_path, encoding='cp1252')
        if "Country" not in cr_box_df.columns:
            raise ValueError("CR Box CSV must have a 'Country' column")
        cr_box_df["Country"] = cr_box_df["Country"].apply(Country._cc.convert, to="name_short")
    
    # ---------------- ECW CSV ----------------
    ecw_df = None
    if ecw_csv_path:
        ecw_df = pd.read_csv(ecw_csv_path, encoding='cp1252')
        required_ecw_cols = ['Country Name', 'Country Code']
        for col in required_ecw_cols:
            if col not in ecw_df.columns:
                raise ValueError(f"ECW CSV must have a column named '{col}'")
    
    # ---------------- Baghouse Airflow CSV ----------------
    baghouse_df = None
    if baghouse_csv_path:
        baghouse_df = pd.read_csv(baghouse_csv_path, encoding='cp1252')
        required_baghouse_cols = ['Country', 'Operating MW']
        for col in required_baghouse_cols:
            if col not in baghouse_df.columns:
                raise ValueError(f"Baghouse CSV must have a column named '{col}'")
        # Standardize country names
        baghouse_df["Country"] = baghouse_df["Country"].apply(Country._cc.convert, to="name_short")
    
    countries = {}
    
    for _, row in df.iterrows():
        iso_code = row['ISO-3']
        country_name = row['Country Name']
        
        # Create country object
        c = Country(name=iso_code)
        c.properties['ISO-3'] = iso_code
        
        # ---------------- Merge CR Box properties ----------------
        if cr_box_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            cr_row = cr_box_df[cr_box_df["Country"] == standardized_name]
            if not cr_row.empty:
                for col in cr_row.columns:
                    if col != "Country":
                        c.properties[col] = cr_row.iloc[0][col]
            else:
                for col in cr_box_df.columns:
                    if col != "Country":
                        c.properties[col] = 0
        
        # ---------------- Merge ECW properties ----------------
        if ecw_df is not None:
            ecw_row = ecw_df[ecw_df["Country Code"] == iso_code]
            if not ecw_row.empty:
                for col in ecw_row.columns:
                    if col not in ["Country Code", "Country Name"]:
                        c.properties[col] = ecw_row.iloc[0][col]
        
        # ---------------- Merge Baghouse properties ----------------
        if baghouse_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            baghouse_row = baghouse_df[baghouse_df["Country"] == standardized_name]
            if not baghouse_row.empty:
                c.properties["Baghouse Operating MW"] = baghouse_row.iloc[0]["Operating MW"]
            else:
                c.properties["Baghouse Operating MW"] = 0
        
        countries[iso_code] = c
    
    return countries


In [3]:
## ---------------------------- Varaibles ---------------------------- 

CR_Box_CADR_LS = 126.13
Filter_Life_Span = ufloat((2-1)/2 , (2-1)/4)
Scale_Up_Factor = 1/0.7
Initial_Stock_in_weeks = ufloat(6,1)
Factory_minimum_production = 50000
Coalbaghouse_efficency = ufloat((0.8+0.5)/2, (0.8-0.5)/4)
Scale_Up_Time_Period_In_Weeks = 52
Coalbaghouse_gradient = ufloat(1717, 419.3/2)
Coalbaghouse_offset = 33807
Coalbaghouse_Utilisation = ufloat(0.3,0.1)


In [10]:
## ---------------------------- CR Box Reference ---------------------------- 

Ind_Market_Rev_Per_MERV = {
    '17-20' : 2208.7e6,
    '5-8'   : 563.4e6,
    '9-12'  : 1271.8e6,
    '1-4'   : 171.7e6,
    '13-16' : 1878.1e6}

Tot_Ind_Air_Filter = Ind_Market_Rev_Per_MERV['17-20']+Ind_Market_Rev_Per_MERV['5-8']+Ind_Market_Rev_Per_MERV['1-4']+Ind_Market_Rev_Per_MERV['9-12']+Ind_Market_Rev_Per_MERV['13-16']

Tot_Air_Filter = 20.8303e9

All_Market_Rev_Per_MERV = {
    '17-20' : Ind_Market_Rev_Per_MERV['17-20']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '5-8'   : Ind_Market_Rev_Per_MERV['5-8']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '9-12'  : Ind_Market_Rev_Per_MERV['9-12']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '1-4'   : Ind_Market_Rev_Per_MERV['1-4']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '13-16' : Ind_Market_Rev_Per_MERV['13-16']*Tot_Air_Filter/Tot_Ind_Air_Filter}

Price_Per_Filter = {
    '1-4'   : ufloat(1031.59,   107.26/2),
    '5-8'   : ufloat(1133.85,   447.48/2),
    '9-12'  : ufloat(1302.51,   554.53/2),
    '13-16' : ufloat(1951.25,   593.63/2),
    '17-20' : ufloat(22925.29,  3740.81/2)}

Volume_to_Sale = 0.508*0.508*0.0254

Sales = {
    '1-4'   : All_Market_Rev_Per_MERV['1-4']/(Price_Per_Filter['1-4']*Volume_to_Sale),
    '5-8'   : All_Market_Rev_Per_MERV['5-8']/(Price_Per_Filter['5-8']*Volume_to_Sale),
    '9-12'  : All_Market_Rev_Per_MERV['9-12']/(Price_Per_Filter['9-12']*Volume_to_Sale),
    '13-16' : All_Market_Rev_Per_MERV['13-16']/(Price_Per_Filter['13-16']*Volume_to_Sale),
    '17-20' : All_Market_Rev_Per_MERV['17-20']/(Price_Per_Filter['17-20']*Volume_to_Sale)}

Panel_Filter = ufloat(0.35,     0.35*0.1/2)

Usable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Scale_Up_Factor

Repurposeable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Filter_Life_Span

In [12]:
## ---------------------------- Countries ---------------------------- 

if "countries_dict" not in globals():
    countries_dict = generate_countries_from_multiple_csvs(country_list_csv, CR_Box_Countries, ecw_country, Baghouse_Airflow)

# ----- CR Box Scale Factor -----
sum_scale = 0
for country in countries_dict.values():
    msa = country.properties["MSA"]
    mva = country.properties["MVA"]
    if msa == 1:
        country.properties["Big_6"] = True
        sum_scale += mva
    else:
        country.properties["Big_6"] = False
scale = Usable_Filters/sum_scale

# ----- Country Calculations -----
for country in countries_dict.values():
    # --- CR Box Manufacturing ---
    msa = country.properties["MSA"]
    mva = country.properties["MVA"]
    x = scale * msa * mva if mva > 55.5 else ufloat(0,0)
    if x.nominal_value < Factory_minimum_production:
        x = ufloat(0,0)
    else:
        x = x/4
    country.properties["CR Box Annual Production"] = ufloat(math.floor(x.nominal_value), x.std_dev)
    country.properties["CR Box Weekly Production"] = country.properties["CR Box Annual Production"] / 52
    country.properties["CADR: CR Box Annual Production"] = country.properties.get("CR Box Annual Production", 0) * CR_Box_CADR_LS
    country.properties["CADR: CR Box Weekly Production"] = country.properties["CADR: CR Box Annual Production"] / 52

    # --- CR Box Initial Stock ---
    if country.properties["Big_6"] == True:
        country.properties["CR Box Initial Stock"] = country.properties["CR Box Weekly Production"]*0.7 * Initial_Stock_in_weeks
        country.properties["CADR: CR Box Initial Stock"] = x*0.7*CR_Box_CADR_LS

    else: 
        country.properties["CR Box Initial Stock"] = ufloat(0,0)
        country.properties["CADR: CR Box Initial Stock"] = ufloat(0,0)

    # --- Delay Function ---
    # --- Manufacturing Delay ---
    m = country.properties["MFS"]
    if m >= 90:
        country.properties['Manufacturing Distribution Delay'] = 1
    elif 55.5 <= m < 90:
        country.properties['Manufacturing Distribution Delay'] = (443/23) - (14/69) * m
    else:
        country.properties['Manufacturing Distribution Delay'] = 0
    # --- Repurposing and Initial Stock Delay ---
    country.properties['Repurposing and Initial Stock Delay'] = 0

    # --- Repurposing of CR Boxes---
    global_MVA=0
    for c in countries_dict.values():
        global_MVA = global_MVA+ c.properties["MVA"]
    country.properties['Ave ECW %'] = ufloat((country.properties["%ECW ILO"] + country.properties["%ECW Poll"] )/2,(country.properties["%ECW ILO"] - country.properties["%ECW Poll"] )/4)
    rel_mva = country.properties["MVA"]/global_MVA
    country.properties['CR Box Repurposing'] = Repurposeable_Filters * rel_mva * country.properties['Ave ECW %']
    country.properties['CADR: CR Box Repurposing'] = country.properties['CR Box Repurposing']*CR_Box_CADR_LS

    # --- Coalbaghouse Calculations ---
    country.properties['Coal Baghouse Airflow Annual'] = Coalbaghouse_efficency*Coalbaghouse_Utilisation*(Coalbaghouse_gradient * country.properties['Baghouse Operating MW'] + Coalbaghouse_offset)
    
countries_dict['CHN'].summary()

--- China ---
ISO-3: CHN
MFS: 95.5
MVA: 4658790000000.0
MSA: 1.0
Labour Force (2024): 774000000.0
Region: Eastern Asia
%ECW ILO: 0.3667734294194942
%ECW Poll: 0.1526772716333544
ECW ILO: 283882634.37068856
ECW Poll: 118172208.24421635
Baghouse Operating MW: 1189041.0
Big_6: True
CR Box Annual Production: (2.8+/-0.4)e+07
CR Box Weekly Production: (5.3+/-0.8)e+05
CADR: CR Box Annual Production: (3.5+/-0.5)e+09
CADR: CR Box Weekly Production: (6.7+/-1.0)e+07
CR Box Initial Stock: (2.2+/-0.5)e+06
CADR: CR Box Initial Stock: (2.4+/-0.4)e+09
Manufacturing Distribution Delay: 1
Repurposing and Initial Stock Delay: 0
Ave ECW %: 0.26+/-0.05
CR Box Repurposing: (7+/-4)e+06
CADR: CR Box Repurposing: (8+/-5)e+08
CADR: Coal Baghouse Annual: (4.0+/-1.5)e+08
Coal Baghouse Airflow Annual: (4.0+/-1.5)e+08


In [8]:
## ---------------------------- Scale Up ---------------------------- 

output_path = BASE_DIR / "results" / "Scale_up_output_MS.csv"

def scale_up(country,t):
    i = 1
    data_point = [0]
    while i <= t:
        prev = data_point[-1]
        if prev == 0:
            prev = country.properties["CADR: CR Box Initial Stock"] + country.properties["CADR: CR Box Repurposing"]
            data_point[-1] = prev
        if country.properties["Manufacturing Distribution Delay"] == None:
            next = 0
        if i<country.properties["Manufacturing Distribution Delay"]:
            next = 0
        else:
            next = prev+ country.properties["CADR: CR Box Weekly Production"]
        data_point.append(next)
        i=i+1
    return data_point

scale_up_data ={}
for country in countries_dict.values():
    data_points = scale_up(country, Scale_Up_Time_Period_In_Weeks)
    scale_up_data[country.name] = data_points

df = pd.DataFrame(scale_up_data).T
df.index.name = "Country"

df.to_csv(output_path, index=True)